<div align="center">

# 👕 K-Nearest Neighbors Classification Using the Fashion-MNIST Dataset

### Fashion-MNIST Dataset

*Image Classification using K-Nearest Neighbors with different K values*

</div>

---

## 📚 1. Import Required Libraries

---

In [2]:
import numpy as np
import time

from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier

from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)

## 🎯 2. Load the Fashion-MNIST Dataset

*Fashion-MNIST contains 70,000 grayscale images of clothing items, where each image consists of 784 pixel features.*

In [3]:
fashion_mnist = fetch_openml(
    "Fashion-MNIST",
    version=1,
    as_frame=False
)

X = fashion_mnist.data
y = fashion_mnist.target.astype(int)

print(f"Total dataset size: {X.shape[0]} images, each with {X.shape[1]} pixels.")

Total dataset size: 70000 images, each with 784 pixels.


## 📦 3. Create a Stratified Dataset Subset

*12,000 images are selected from the complete dataset while preserving the distribution of the ten clothing classes.*

In [4]:
X_subset, _, y_subset, _ = train_test_split(
    X,
    y,
    train_size=12000,
    stratify=y,
    random_state=42
)

## ✂️ 4. Split the Dataset into Training and Testing Sets

*The selected dataset is divided into 10,000 training images and 2,000 testing images using stratified sampling.*

In [5]:
X_train, X_test, y_train, y_test = train_test_split(
    X_subset,
    y_subset,
    test_size=2000,
    stratify=y_subset,
    random_state=42
)

print(f"Training images: {X_train.shape[0]}")
print(f"Testing images: {X_test.shape[0]}")

Training images: 10000
Testing images: 2000


## ⚖️ 5. Normalize the Pixel Values

*Pixel values originally range from 0 to 255. Scaling them to the range 0 to 1 improves distance-based calculations used by KNN.*

In [6]:
print(f"Before scaling -> Min: {X_train.min()}, max: {X_train.max()}")

X_train = X_train / 255.0
X_test = X_test / 255.0

print(f"After scaling -> Min: {X_train.min()}, max: {X_train.max()}")

Before scaling -> Min: 0, max: 255
After scaling -> Min: 0.0, max: 1.0


## 🤖 6. Train KNN Models with Different K Values

*The K-Nearest Neighbors classifier is trained using multiple K values to study their effect on classification accuracy and prediction time.*

### K Values

*The following values of K are evaluated: 1, 3, 5, 7, 9, and 15.*

In [7]:
k_values = [1, 3, 5, 7, 9, 15]
results = {}

print(f"{'K value':<8} | {'Accuracy':<10} | {'Prediction Time (seconds)':<25}")
print("_" * 55)

for k in k_values:
    knn = KNeighborsClassifier(
        n_neighbors=k,
        metric="euclidean",
        n_jobs=1
    )

    knn.fit(X_train, y_train)

    start_time = time.time()
    y_pred = knn.predict(X_test)
    elapsed_time = time.time() - start_time

    acc = accuracy_score(y_test, y_pred)

    results[k] = {
        "accuracy": acc,
        "time": elapsed_time,
        "predictions": y_pred
    }

    print(f"{k:<8} | {acc * 100:<9.2f}% | {elapsed_time:<25.2f}")

K value  | Accuracy   | Prediction Time (seconds)
_______________________________________________________
1        | 79.30    % | 2.42                     
3        | 80.90    % | 0.20                     
5        | 81.00    % | 0.21                     
7        | 81.10    % | 0.21                     
9        | 80.95    % | 0.20                     
15       | 80.25    % | 0.21                     


## 📊 7. Identify the Best K Value

*The K value producing the highest classification accuracy is selected as the best-performing KNN configuration.*

In [8]:
best_k = max(
    results,
    key=lambda k: results[k]["accuracy"]
)

print(
    f"Best K is: {best_k} "
    f"with {results[best_k]['accuracy'] * 100:.2f}% accuracy"
)

Best K is: 7 with 81.10% accuracy


## 🏷️ 8. Define Fashion-MNIST Class Labels

*The numerical class labels are mapped to their corresponding clothing categories for easier interpretation of the classification results.*

In [9]:
class_names = [
    "T-shirt/top",
    "Trouser",
    "Pullover",
    "Dress",
    "Coat",
    "Sandal",
    "Shirt",
    "Sneaker",
    "Bag",
    "Ankle boot"
]

## 📋 9. Generate the Classification Report

*The best-performing KNN model is evaluated using precision, recall, F1-score, and overall accuracy for each clothing category.*

In [10]:
print("Per-Class Classification Report:\n")

print(
    classification_report(
        y_test,
        results[best_k]["predictions"],
        target_names=class_names
    )
)

Per-Class Classification Report:

              precision    recall  f1-score   support

 T-shirt/top       0.73      0.83      0.78       200
     Trouser       0.97      0.94      0.96       200
    Pullover       0.67      0.73      0.70       200
       Dress       0.85      0.83      0.84       200
        Coat       0.74      0.69      0.72       200
      Sandal       1.00      0.76      0.86       200
       Shirt       0.58      0.55      0.56       200
     Sneaker       0.81      0.92      0.86       200
         Bag       0.97      0.94      0.95       200
  Ankle boot       0.85      0.94      0.89       200

    accuracy                           0.81      2000
   macro avg       0.82      0.81      0.81      2000
weighted avg       0.82      0.81      0.81      2000

